# Optimizer Arena: Gradient Descent vs Newton vs PyTorch

This notebook walks through the whole project in order:

1. Synthetic data with a known answer (b = 2, w = 3)
2. Gradient descent and Newton's method, written from scratch with NumPy
3. The loss surface and the paths the two methods take
4. The learning rate experiment
5. Real data (California Housing) and why feature scaling matters
6. Comparison with PyTorch (`torch.optim.SGD`, `Adam`, `LBFGS`) and autograd

All logic lives in `src/optimizer_arena/`. The notebook only calls it, plots and comments.

**Setup:** run `jupyter lab` from the project root with the virtual environment active. See the README for details.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

import numpy as np
import matplotlib.pyplot as plt

from optimizer_arena.data import (
    add_bias_column, load_california_housing, make_linear_data, standardize,
)
from optimizer_arena.models import gradient, hessian, mse_loss
from optimizer_arena.optimizers import gradient_descent, newton

plt.rcParams["figure.dpi"] = 110

## 1. Synthetic data

We generate points around the line `y = 3x + 2` with Gaussian noise. Because we know the truth, we can check our code.
The fixed seed makes every run identical.

In [ ]:
x, y = make_linear_data(n_samples=100, true_w=3.0, true_b=2.0, noise_std=1.0, seed=42)
X = add_bias_column(x)          # first column of ones, so theta = (b, w)
print("x:", x.shape, " y:", y.shape, " X:", X.shape)

plt.scatter(x, y, s=12)
plt.xlabel("x"); plt.ylabel("y"); plt.title("Synthetic data")
plt.show()

## 2. Gradient descent and Newton

Model and loss, with `theta = (b, w)` and `X = [1, x]`:

$$\hat y = X\theta, \qquad L(\theta) = \frac1n \lVert X\theta - y\rVert^2, \qquad \nabla L = \frac2n X^\top (X\theta - y), \qquad H = \frac2n X^\top X$$

Updates:

$$\text{GD: } \theta_{k+1} = \theta_k - \alpha \nabla L(\theta_k) \qquad \text{Newton: } \theta_{k+1} = \theta_k - H^{-1}\nabla L(\theta_k)$$

The exact least squares solution (normal equation) is our answer key.

In [ ]:
exact = np.linalg.solve(X.T @ X, X.T @ y)
theta_gd, hist_gd = gradient_descent(X, y, lr=0.01, n_iters=1000)
theta_nt, hist_nt = newton(X, y)

print("Exact   :", exact)
print("GD      :", theta_gd, " steps:", len(hist_gd) - 1)
print("Newton  :", theta_nt, " steps:", len(hist_nt) - 1)
print("Condition number of H:", np.linalg.cond(hessian(X)))

Newton lands on the exact solution in one step because the MSE loss is quadratic and its Hessian is constant.
Gradient descent gets close but is still slightly off after 1000 steps, because the loss surface is a long, narrow valley (condition number about 137).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hist_gd, label="Gradient descent")
ax.plot(hist_nt, "o-", label="Newton")
ax.set_yscale("log"); ax.set_xlabel("Iteration"); ax.set_ylabel("Loss (MSE, log scale)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.show()

## 3. Loss surface and optimizer paths

Contour lines connect points with the same loss. Where lines are close together, the surface is steep.

In [ ]:
start = np.array([-4.0, -1.0])
_, _, path_gd = gradient_descent(X, y, lr=0.025, n_iters=80, theta0=start, return_path=True)
_, _, path_nt = newton(X, y, theta0=start, return_path=True)

bs = np.linspace(-5, 5, 150)
ws = np.linspace(-2, 12, 150)
B, W = np.meshgrid(bs, ws)
Z = np.mean((B[..., None] + W[..., None] * x - y) ** 2, axis=-1)

fig, ax = plt.subplots(figsize=(7, 6))
ax.contour(B, W, Z, levels=np.logspace(np.log10(Z.min()), np.log10(Z.max()), 25), cmap="viridis")
ax.plot(path_gd[:, 0], path_gd[:, 1], "o-", ms=3, label="Gradient descent")
ax.plot(path_nt[:, 0], path_nt[:, 1], "o-", ms=4, label="Newton")
ax.plot(*exact, "k*", ms=15, label="Exact minimum")
ax.set_xlabel("b (intercept)"); ax.set_ylabel("w (slope)"); ax.legend()
plt.show()

## 4. Learning rate experiment

Gradient descent is stable only if `alpha < 2 / lambda_max`, where `lambda_max` is the largest eigenvalue of the Hessian.

In [ ]:
lam_max = np.linalg.eigvalsh(hessian(X)).max()
print("Stability limit 2 / lambda_max =", 2 / lam_max)

fig, ax = plt.subplots(figsize=(7, 4))
for lr, label in [(0.0005, "too small"), (0.01, "good"), (0.035, "too large")]:
    _, hist = gradient_descent(X, y, lr=lr, n_iters=200)
    ax.plot(hist, label=f"{label} ({lr})")
ax.set_yscale("log"); ax.set_xlabel("Iteration"); ax.set_ylabel("Loss (MSE, log scale)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.show()

## 5. Real data: California Housing and feature scaling

20,640 houses, 8 features, target in units of $100k. Raw features live on very different scales (income 1 to 15, population in the thousands),
which makes the loss surface an extremely narrow valley. Standardizing each feature fixes that.

The first call downloads the data (needs internet once).

In [ ]:
features, y_cal = load_california_housing()

results = {}
for name, F in [("raw", features), ("standardized", standardize(features))]:
    Xc = add_bias_column(F)
    H = hessian(Xc)
    lr = 1.0 / np.linalg.eigvalsh(H).max()
    exact_c = np.linalg.solve(Xc.T @ Xc, Xc.T @ y_cal)
    _, h_gd = gradient_descent(Xc, y_cal, lr=lr, n_iters=1000)
    _, h_nt = newton(Xc, y_cal)
    results[name] = (h_gd, h_nt)
    print(f"--- {name} ---")
    print("condition number:", f"{np.linalg.cond(H):.3g}", " lr:", f"{lr:.3g}")
    print("GD loss after 1000 steps:", h_gd[-1])
    print("Newton loss after", len(h_nt) - 1, "step(s):", h_nt[-1])
    print("Exact loss:", np.mean((Xc @ exact_c - y_cal) ** 2))

fig, ax = plt.subplots(figsize=(8, 4.5))
for name, (h_gd, h_nt) in results.items():
    ax.plot(h_gd, label=f"GD ({name})")
    ax.plot(h_nt, "o-", label=f"Newton ({name})")
ax.set_yscale("log"); ax.set_xlabel("Iteration"); ax.set_ylabel("Loss (MSE, log scale)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.show()

## 6. Comparison with PyTorch

PyTorch is optional for the rest of the project. Install it with `pip install torch`.

We check three things on the standardized data:

1. **Autograd vs our formulas:** PyTorch computes the gradient and Hessian automatically. They should match our hand-derived ones.
2. **`torch.optim.SGD` vs our gradient descent:** with full batches and the same learning rate, they are the same algorithm.
3. **`Adam` and `LBFGS`:** other optimizers on the same problem.

In [ ]:
import torch

Xs = add_bias_column(standardize(features))   # float64
exact_s = np.linalg.solve(Xs.T @ Xs, Xs.T @ y_cal)
lr_s = 1.0 / np.linalg.eigvalsh(hessian(Xs)).max()
Xt, yt = torch.from_numpy(Xs), torch.from_numpy(y_cal)

def torch_mse(theta):
    return ((Xt @ theta - yt) ** 2).mean()

# 1. autograd vs hand-derived
theta0 = np.random.default_rng(0).normal(size=Xs.shape[1])
t = torch.tensor(theta0, requires_grad=True)
torch_mse(t).backward()
print("max |autograd grad - our grad|   :", np.abs(t.grad.numpy() - gradient(Xs, y_cal, theta0)).max())
H_auto = torch.autograd.functional.hessian(torch_mse, torch.tensor(theta0)).numpy()
print("max |autograd Hessian - our H|   :", np.abs(H_auto - hessian(Xs)).max())

In [ ]:
def run_torch(make_optimizer, n_iters=1000):
    theta = torch.zeros(Xs.shape[1], dtype=torch.float64, requires_grad=True)
    opt = make_optimizer([theta])
    hist = []
    for _ in range(n_iters):
        opt.zero_grad()
        loss = torch_mse(theta)
        hist.append(loss.item())
        loss.backward()
        opt.step()
    hist.append(torch_mse(theta).item())
    return theta.detach().numpy().copy(), hist

theta_ours, hist_ours = gradient_descent(Xs, y_cal, lr=lr_s, n_iters=1000)
theta_sgd, hist_sgd = run_torch(lambda p: torch.optim.SGD(p, lr=lr_s))
theta_adam, hist_adam = run_torch(lambda p: torch.optim.Adam(p, lr=0.05))
theta_newton, hist_newton = newton(Xs, y_cal)

print("max |theta_ours - theta_torch_SGD| :", np.abs(theta_ours - theta_sgd).max())
print()
print(f"{'method':<26}{'final loss':>14}{'max|theta-exact|':>20}")
for name, th, h in [("Ours: GD", theta_ours, hist_ours), ("PyTorch SGD", theta_sgd, hist_sgd),
                    ("PyTorch Adam", theta_adam, hist_adam), ("Ours: Newton", theta_newton, hist_newton)]:
    print(f"{name:<26}{h[-1]:>14.6f}{np.abs(th - exact_s).max():>20.2e}")

In [ ]:
theta = torch.zeros(Xs.shape[1], dtype=torch.float64, requires_grad=True)
opt = torch.optim.LBFGS([theta], lr=1.0, max_iter=20, history_size=10, line_search_fn="strong_wolfe")

def closure():
    opt.zero_grad()
    loss = torch_mse(theta)
    loss.backward()
    return loss

hist_lbfgs = [torch_mse(theta).item()]
for _ in range(10):
    opt.step(closure)
    hist_lbfgs.append(torch_mse(theta).item())

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(hist_ours, lw=3, alpha=0.5, label="Ours: GD")
ax.plot(hist_sgd, "--", label="PyTorch SGD")
ax.plot(hist_adam, label="PyTorch Adam")
ax.plot(hist_newton, "o-", label="Ours: Newton")
ax.plot(hist_lbfgs, "s:", label="PyTorch LBFGS (per outer step)")
ax.set_yscale("log"); ax.set_xlabel("Update step"); ax.set_ylabel("Loss (MSE, log scale)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.show()

## Conclusions

- Our gradient and Hessian formulas are correct: PyTorch autograd agrees with them.
- `torch.optim.SGD` with full batches is exactly our gradient descent. Frameworks do not use different math, they automate the gradient.
- Newton reaches the exact solution in one step for linear regression, but needs to solve a d x d system each step.
- Gradient descent depends on the problem scale. Standardizing the features lowered the condition number from about 5.7e10 to 44.
- LBFGS approximates the Hessian from past gradients, a middle path between GD and Newton.